# PRCP-1010 — Insurance Claim Prediction (InsClaimPred)

**Domain:** Finance / Insurance  
**Level:** Intern / Capstone starter project

## Problem Statement
1. **Task 1:** Build a predictive model that helps the insurance marketing team identify which customers are more likely to engage with / buy the product (claim-risk related outcome in this dataset).
2. **Task 2:** Provide actionable suggestions to the marketing team to improve product uptake.

> Due to privacy, feature names are anonymized. This notebook focuses on modeling, model comparison, recommendations, and challenges.

**Dataset:** 595,212 rows × 59 columns (`id`, `target`, + 57 anonymized features).  
**Target:** binary (`0` = no claim / majority, `1` = claim / rare event ~3.6%).


## 0. Setup & Imports

In [ ]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
    RocCurveDisplay,
)

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

ROOT = Path("..").resolve() if Path("..").joinpath("data").exists() else Path(".").resolve()
DATA_PATH = ROOT / "data" / "train.csv"
REPORTS = ROOT / "reports"
REPORTS.mkdir(parents=True, exist_ok=True)
print("Project root:", ROOT)
print("Data path:", DATA_PATH)

## 1. Load Data

If `data/train.csv` is missing, run:

```bash
python scripts/download_data.py
```

For faster iteration on a laptop / intern workstation we use a **stratified sample**. Set `SAMPLE_SIZE = None` to use the full dataset.


In [ ]:
SAMPLE_SIZE = 80000  # set to None for full 595k rows

if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Missing {DATA_PATH}. Run: python scripts/download_data.py"
    )

df_full = pd.read_csv(DATA_PATH)
print("Full dataset shape:", df_full.shape)

if SAMPLE_SIZE is not None and SAMPLE_SIZE < len(df_full):
    parts = []
    for _, group in df_full.groupby("target"):
        n = max(1, int(round(SAMPLE_SIZE * len(group) / len(df_full))))
        n = min(n, len(group))
        parts.append(group.sample(n=n, random_state=RANDOM_STATE))
    df = pd.concat(parts).sample(frac=1.0, random_state=RANDOM_STATE).reset_index(drop=True)
else:
    df = df_full.copy()

print("Working dataset shape:", df.shape)
df.head()

## 2. Quick Data Understanding (light EDA)

The project brief says EDA can be skipped because features are anonymized. Still, as interns we do a **minimal check**: shape, target balance, dtypes, and missing-value sentinel `-1`.


In [ ]:
print("Columns:", list(df.columns))
print("\nDtypes:\n", df.dtypes.value_counts())
print("\nTarget counts:\n", df["target"].value_counts())
print("\nTarget rate:\n", df["target"].value_counts(normalize=True))

missing_counts = (df == -1).sum()
missing_counts = missing_counts[missing_counts > 0].sort_values(ascending=False)
print("\nFeatures with -1 missing sentinel:\n", missing_counts)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
df["target"].value_counts().plot(kind="bar", ax=axes[0], color=["#4C72B0", "#DD8452"])
axes[0].set_title("Target Distribution (counts)")
axes[0].set_xlabel("target")
axes[0].set_ylabel("count")

if len(missing_counts):
    missing_counts.head(10).plot(kind="barh", ax=axes[1], color="#55A868")
    axes[1].set_title("Top missing features (-1 count)")
else:
    axes[1].set_title("No -1 missing values found")
plt.tight_layout()
plt.show()

## 3. Preprocessing

- Replace `-1` with `NaN`
- Drop `id` (identifier) and `target` from features
- Impute numerics with median inside each model pipeline
- Scale features for linear models


In [ ]:
work = df.replace(-1, np.nan)
y = work["target"].astype(int)
feature_cols = [c for c in work.columns if c not in ("id", "target")]
X = work[feature_cols]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)
print(X_train.shape, X_test.shape)
print("Train positive rate:", y_train.mean())
print("Test positive rate:", y_test.mean())

## 4. Model Training

We compare a few classic intern-level baselines that handle imbalance differently:

| Model | Why include it |
|---|---|
| Logistic Regression | Strong, interpretable linear baseline |
| Random Forest | Non-linear interactions, class weights |
| Gradient Boosting | Often strong tabular performance |
| XGBoost (if installed) | Popular boosting baseline for competitions |


In [ ]:
models = {
    "Logistic Regression": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("clf", LogisticRegression(
            max_iter=1000,
            class_weight="balanced",
            random_state=RANDOM_STATE,
        )),
    ]),
    "Random Forest": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("clf", RandomForestClassifier(
            n_estimators=150,
            max_depth=12,
            min_samples_leaf=20,
            class_weight="balanced_subsample",
            n_jobs=-1,
            random_state=RANDOM_STATE,
        )),
    ]),
    "Gradient Boosting": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("clf", GradientBoostingClassifier(
            n_estimators=120,
            learning_rate=0.08,
            max_depth=3,
            random_state=RANDOM_STATE,
        )),
    ]),
}

try:
    from xgboost import XGBClassifier
    neg = int((y_train == 0).sum())
    pos = int((y_train == 1).sum())
    models["XGBoost"] = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("clf", XGBClassifier(
            n_estimators=200,
            max_depth=4,
            learning_rate=0.08,
            subsample=0.9,
            colsample_bytree=0.9,
            scale_pos_weight=neg / max(pos, 1),
            eval_metric="auc",
            n_jobs=-1,
            random_state=RANDOM_STATE,
        )),
    ])
    print("XGBoost available — included in comparison.")
except ImportError:
    print("XGBoost not installed — skipping.")

def normalized_gini(y_true, y_score):
    return 2 * roc_auc_score(y_true, y_score) - 1

trained = {}
rows = []

for name, model in models.items():
    print(f"\n=== Training {name} ===")
    model.fit(X_train, y_train)
    proba = model.predict_proba(X_test)[:, 1]
    pred = (proba >= 0.5).astype(int)
    metrics = {
        "model": name,
        "roc_auc": roc_auc_score(y_test, proba),
        "normalized_gini": normalized_gini(y_test, proba),
        "avg_precision": average_precision_score(y_test, proba),
        "accuracy": accuracy_score(y_test, pred),
        "precision": precision_score(y_test, pred, zero_division=0),
        "recall": recall_score(y_test, pred, zero_division=0),
        "f1": f1_score(y_test, pred, zero_division=0),
    }
    print(classification_report(y_test, pred, zero_division=0))
    print({k: round(v, 4) if isinstance(v, float) else v for k, v in metrics.items()})
    trained[name] = {"model": model, "proba": proba, "pred": pred, "metrics": metrics}
    rows.append(metrics)

comparison = pd.DataFrame(rows).sort_values("roc_auc", ascending=False)
comparison

## 5. Model Comparison Report

In [ ]:
display(comparison)

comparison.to_csv(REPORTS / "model_comparison.csv", index=False)

plt.figure(figsize=(8, 6))
for name, payload in trained.items():
    RocCurveDisplay.from_predictions(
        y_test,
        payload["proba"],
        name=f"{name} (AUC={payload['metrics']['roc_auc']:.3f})",
        ax=plt.gca(),
    )
plt.plot([0, 1], [0, 1], "k--", label="Random")
plt.title("ROC Curve Comparison")
plt.legend(loc="lower right")
plt.tight_layout()
plt.savefig(REPORTS / "roc_curves.png", dpi=150)
plt.show()

best_name = comparison.iloc[0]["model"]
best = trained[best_name]
print("Best model for production (by ROC-AUC):", best_name)
print(comparison.iloc[0].to_dict())

cm = confusion_matrix(y_test, best["pred"])
plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")
plt.title(f"Confusion Matrix — {best_name}")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.tight_layout()
plt.savefig(REPORTS / f"confusion_{best_name.lower().replace(' ', '_')}.png", dpi=150)
plt.show()

### Production recommendation

Prefer the model with the **highest ROC-AUC / Normalized Gini** on a held-out stratified test set (or cross-validation in a follow-up).

For an imbalanced insurance dataset:
- **Do not** choose the model with highest accuracy alone (always predicting 0 is ~96% accurate).
- Prefer ranking metrics (`ROC-AUC`, `Normalized Gini`, `Average Precision`).
- In production, use predicted **probabilities** to prioritize outreach / underwriting review.


## 6. Feature Importance (for marketing intuition)

Even with anonymized names, relative importance helps marketing / product teams know *which groups of signals* matter most.


In [ ]:
best_model = best["model"]
clf = best_model.named_steps["clf"]

if hasattr(clf, "feature_importances_"):
    importances = pd.Series(clf.feature_importances_, index=feature_cols).sort_values(ascending=False)
elif hasattr(clf, "coef_"):
    importances = pd.Series(np.abs(clf.coef_[0]), index=feature_cols).sort_values(ascending=False)
else:
    importances = pd.Series(dtype=float)

if len(importances):
    top_n = importances.head(15)
    plt.figure(figsize=(8, 6))
    top_n.iloc[::-1].plot(kind="barh", color="#4C72B0")
    plt.title(f"Top 15 Feature Importances — {best_name}")
    plt.tight_layout()
    plt.savefig(REPORTS / "feature_importance.png", dpi=150)
    plt.show()
    display(top_n.to_frame("importance"))
else:
    print("This model does not expose a simple importance vector.")

## 7. Suggestions for the Insurance Marketing Team (Task 2)

1. **Score-and-rank customers**: Use model probabilities to sort the book of business. Contact the **top decile** first instead of a cold random list.
2. **Treat the rare event carefully**: Because claim/positive outcome is ~3.6%, a hard 0.5 threshold is often too aggressive or too conservative. Tune the cutoff to the campaign budget and tolerance for false positives.
3. **Offer differentiated products**: For higher-risk scored segments, propose safer package tiers (higher deductible / usage-based add-ons). For lower-risk scored segments, push premium conversion offers.
4. **Measure lift with A/B tests**: Compare conversion when targeting model-selected customers vs. randomly selected customers.
5. **Refresh regularly**: Retrain periodically — customer mix, claims patterns, and product offerings drift.
6. **Combine with business rules**: Keep compliance / eligibility filters on top of the ML ranking so marketing actions remain explainable and fair.


## 8. Report on Challenges Faced

| Challenge | Why it mattered | Technique used |
|---|---|---|
| Severe class imbalance (~3.6% positives) | Accuracy looked high while missing nearly all positives | Used `class_weight` / `scale_pos_weight`; prioritized ROC-AUC, Gini, PR-AUC |
| Missing values encoded as `-1` | Models would treat missing as a real numeric value | Replaced `-1` with `NaN` and median-imputed in pipelines |
| Anonymized feature names | Hard to write rich business narratives from EDA | Focused on modeling quality + feature-importance proxies |
| Large dataset (595k rows) | Slow iteration for an intern workflow | Stratified sampling for development; full-data option documented |
| Metric choice for production | Marketing cares about ranking/outreach efficiency | Recommended probability ranking + campaign lift testing |

### Techniques summary
- Stratified train/test split to preserve class ratio
- Median imputation + scaling for linear models
- Tree/boosting models for non-linear interactions
- Multi-metric comparison report for fair model selection


## 9. Conclusion

This notebook delivers an intern-level baseline for **PRCP-1010-InsClaimPred**:
- Comparable ML models on the insurance claim dataset
- A clear production recommendation based on ranking metrics
- Practical marketing suggestions and a challenges report

**Next steps (stretch goals):** cross-validation, hyperparameter tuning, calibration curves, and deployment of the best model behind a small scoring API.


In [ ]:
# Final summary table for submission
final_summary = {
    "best_model": best_name,
    "roc_auc": round(best["metrics"]["roc_auc"], 4),
    "normalized_gini": round(best["metrics"]["normalized_gini"], 4),
    "avg_precision": round(best["metrics"]["avg_precision"], 4),
    "rows_used": int(len(df)),
    "n_features": int(len(feature_cols)),
}
print(final_summary)
comparison